# 9c · Independent verification — a **non-LLM** entailment checker

nb9 and nb9b both answer "is this assertion supported?" with **one Gemini judge**, and its reliability
is unmeasured. Every factuality number this pipeline reports therefore rests on a single closed model
whose failure modes we cannot inspect, cannot pin to a version, and cannot reproduce once the endpoint
changes. That is the weakest link in the methodology.

This notebook adds a **second, independent verifier of a completely different kind**: a small
encoder-only entailment model that runs locally. Given a premise (the article) and a hypothesis (the
verbalised assertion), it answers *does the premise support the hypothesis?* — the exact task
NLI / fact-verification models were built for.

**Why this is worth a section in the thesis**

| | Gemini judge (nb9/nb9b) | entailment checker (this) |
|---|---|---|
| kind | autoregressive LLM, closed weights | encoder classifier, open weights (MIT) |
| cost | per call, metered | free after a one-time download |
| determinism | none (endpoint may change under you) | exact — same graph always scores the same |
| provenance | a natural-language justification | a probability + the sentence that maximises it |

The two are wrong in *different* ways, which is the whole point: an encoder trained on entailment does
not share an LLM's tendency to be agreeable, and an LLM does not share an encoder's brittleness to
paraphrase. So:

- **where they agree** — a far stronger claim than either alone, and the number to report;
- **where they disagree** — precisely the set worth reading by hand, and usually a small one.

**Verifier choice (checked and *tested* August 2026 — this area moves, and the published ranking is
not the whole story).** On the [LLM-AggreFact](https://llm-aggrefact.github.io/) benchmark the best
sub-1B model is `FactCG-DeBERTa-v3-Large` (0.4B, 75.6 avg balanced accuracy), ahead of MiniCheck at
the same scale. **Do not use its HuggingFace `model.safetensors` export**: it loads without any
missing-key warning but its classification head is untrained, and it scores true and false claims
identically (~0.79–0.88 for everything, AUC ≈ 0.53 on unambiguous probe data — verified here, not
assumed). Its real weights ship separately as a 5.2 GB PyTorch-Lightning checkpoint that needs the
authors' own wrapper class.

So the default is **`MiniCheck-DeBERTa-v3-Large`** (0.4B, MIT, EMNLP 2024), which loads correctly as
a plain `AutoModelForSequenceClassification` and separates the same probe cleanly (0.91 / 0.01 /
0.96 / 0.01). It gives up ~1.6 points of benchmark score for a model that demonstrably works.

`Bespoke-MiniCheck-7B` tops the board overall (77.4) but is a 7B LLM, which would defeat the purpose
of an independent *non-LLM* check. **AlignScore** (2023) and **SummaC** (2021) are the previous
generation of this idea and are now clearly behind — cite them as lineage, don't run them as the
verifier. Section 2 re-tests whatever you point this at, so swapping models stays safe.

Reads  `results/<model>/<id>/e2e/grounding_judgments.json` (nb9) and/or
       `grounding_fulltext_judgments.json` (nb9b) + `dataset/articles/<id>.txt`
Writes `results/<model>/<id>/e2e/nli_verdicts.json`
       `results/<model>/_global/nli_agreement.json` + `nli_disagreements.csv`

In [ ]:
# --- resolve Phase2_LLM_Extraction root (works whether this notebook lives at the
# project root or under notebooks/<track>/) so results/ & dataset/ paths are stable.
from pathlib import Path
def _find_root(start: Path = None):
    start = (start or Path.cwd()).resolve()
    for p in (start, *start.parents):
        if (p / 'results').is_dir() and (p / 'dataset').is_dir():
            return p
    return start
ROOT = _find_root()
# make Path writes create missing parent dirs (results/ now has sub-folders like
# two_staged/1_entities/); guarded so re-running the cell never double-wraps.
import pathlib as _pl
if not getattr(_pl.Path.write_text, '_mkparents', False):
    _wt, _wb = _pl.Path.write_text, _pl.Path.write_bytes
    def write_text(self, *a, **k):
        self.parent.mkdir(parents=True, exist_ok=True); return _wt(self, *a, **k)
    def write_bytes(self, *a, **k):
        self.parent.mkdir(parents=True, exist_ok=True); return _wb(self, *a, **k)
    write_text._mkparents = write_bytes._mkparents = True
    _pl.Path.write_text, _pl.Path.write_bytes = write_text, write_bytes

In [ ]:
# ---- WHICH MODEL'S RESULTS ---------------------------------------------------------------
RESULTS_MODEL = "gemini-2.5-flash"

# ---- WHICH ARTICLES -----------------------------------------------------------------------
# Same knob as nb9 / nb9b: None = every article that has a judgment log to verify.
ARTICLES = None
# ARTICLES = ["art1", "art5"]

# ---- WHICH JUDGE TO CHECK -----------------------------------------------------------------
# "fulltext" -> nb9b's log (grounding_fulltext_judgments.json) — the like-for-like comparison, since
#               nb9b's judge also reads the whole article, exactly as the entailment model does.
# "evidence" -> nb9's log (grounding_judgments.json). Note the asymmetry: nb9's judge saw only its
#               evidence bundle while this verifier sees the article, so disagreements there include
#               nb9's window being too narrow — that is a finding, not noise, but read it that way.
# "both"     -> load both logs and report agreement against each separately.
JUDGE_SOURCE = "both"
# ------------------------------------------------------------------------------------------

# --- Verifier ------------------------------------------------------------------------------
# All of these are AutoModelForSequenceClassification with a 2-way head, so they are interchangeable
# here, and section 2 re-verifies whichever one you pick before it is used.
#   lytang/MiniCheck-DeBERTa-v3-Large    0.4B   MIT, EMNLP 2024   <- default: verified working here
#   lytang/MiniCheck-RoBERTa-Large       0.35B  MIT               smaller/faster, weaker on multi-hop
#   MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli      generic NLI head — the honest
#     "off-the-shelf entailment baseline" for an ablation. Not trained on document-grounded fact
#     checking, so expect it to be harsher on claims needing two sentences to support.
#
# NOT yaxili96/FactCG-DeBERTa-v3-Large — it is the better model on paper (best sub-1B on
# LLM-AggreFact) but its safetensors export carries an UNTRAINED classifier head: it loads with no
# missing-key warning and then scores every claim ~0.9 regardless of truth. The trained weights are
# the 5.2 GB pytorch_lightning_ckpt/factcg_dbt.ckpt in the same repo and need the authors' wrapper.
# The section 2 probe catches this (AUC 0.53) and refuses to continue — leave it in place.
VERIFIER_MODEL = "lytang/MiniCheck-DeBERTa-v3-Large"

THRESHOLD   = 0.5    # P(supported) >= this -> supported. Section 6 calibrates this against your
                     # human-labelled sample instead of leaving it at the arbitrary default.
MAX_LEN     = 512    # premise+hypothesis token budget per forward pass
BATCH_SIZE  = 8      # starting batch. DeBERTa-v3's disentangled attention allocates a lot at 512
                     # tokens (an M1 with other apps open OOMs at 16), so this is only a starting
                     # point: section 2's scorer halves it automatically on an out-of-memory error
                     # and keeps the lower value for the rest of the run. Raise it on a real GPU.
WINDOW      = 3      # sentences per sliding window in the localisation pass (see section 3)
SENTENCE_PASS = True # the localisation pass is ~10x the forward passes of the document pass.
                     # Set False for a quick document-only run; you then lose `nli_best_sentence`.
FORCE_RESCORE = False  # True = re-score articles that already have nli_verdicts.json

# --- Paths ---------------------------------------------------------------------------------
TRACK_DIR   = "e2e"
RESULTS_DIR = ROOT / "results" / RESULTS_MODEL
ARTICLE_DIR = ROOT / "dataset" / "articles"
GLOBAL_DIR  = RESULTS_DIR / "_global"
OUT_NAME    = "nli_verdicts.json"

LOGS = {"evidence": "grounding_judgments.json", "fulltext": "grounding_fulltext_judgments.json"}
SOURCES = list(LOGS) if JUDGE_SOURCE == "both" else [JUDGE_SOURCE]

import json, re, time
def _natkey(s):
    return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", str(s))]

def resolve_articles(ids=None):
    """-> article ids that have at least one of the requested judgment logs AND their source text."""
    def usable(a):
        return ((ARTICLE_DIR / f"{a}.txt").exists()
                and any((RESULTS_DIR / a / TRACK_DIR / LOGS[s]).exists() for s in SOURCES))
    if ids is None:
        return sorted((p.name for p in RESULTS_DIR.iterdir() if p.is_dir() and usable(p.name)),
                      key=_natkey)
    missing = [a for a in ids if not usable(a)]
    if missing:
        print(f"!! skipped — no {SOURCES} judgment log (or no article text): {missing}")
    return [a for a in ids if usable(a)]

ARTICLE_IDS = resolve_articles(ARTICLES)
assert ARTICLE_IDS, (f"no article under {RESULTS_DIR} has a {SOURCES} judgment log — "
                     "run 9_grounding.ipynb / 9b_grounding_fulltext.ipynb first")
print(f"{len(ARTICLE_IDS)} article(s): {ARTICLE_IDS}")
print(f"checking judge(s): {SOURCES} | verifier: {VERIFIER_MODEL}")

## 1. Load the verifier

One download (~1.7 GB), then it runs offline and free forever. Inference runs under
`torch.inference_mode()` with the model in `eval()`, so scores are deterministic — re-running this
notebook on the same graph reproduces the same numbers exactly, which is the property the Gemini
judge cannot offer.

The scoring itself (section 4) is the only expensive part, and it does **not** have to run here. On
a laptop GPU it is slow enough to be annoying — an M1 with other apps open drops to a batch size of
2 and takes tens of minutes per article. To run it on Lab-IA instead:

```bash
cd Phase2_LLM_Extraction/cluster && ./nli_sync.sh all      # push ~1 MB, score on an A6000, pull back
```

That writes the same `nli_verdicts.json` files this notebook caches to, using the same
`notebooks/nli_core.py` scoring code, so **sections 5–7 below work unchanged** — they read the
verdict files, need no GPU and no model, and finish instantly. If you go that route you can skip
straight to section 5 after running the config cell.

In [ ]:
%pip install -q torch transformers sentencepiece protobuf

import sys, pathlib, importlib
# nli_core lives next to llm_backend.py / nb_parallel.py, at the notebooks/ root
for _p in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_p / "nli_core.py").exists():
        sys.path.insert(0, str(_p)); break
import torch, nli_core
importlib.reload(nli_core)

# The scoring lives in nli_core so that this notebook and the cluster job (cluster/nli_verify.py)
# run the SAME code. If they each had their own sentence splitter or chunker they would assign
# different probabilities to the same assertion, and every agreement number below would be
# comparing two different measurements.
VERIFIER = nli_core.Verifier(VERIFIER_MODEL, max_len=MAX_LEN, batch_size=BATCH_SIZE)
print(f"{VERIFIER_MODEL} on {VERIFIER.device}  ({VERIFIER.n_params/1e6:.0f}M params)")
print("labels:", VERIFIER.model.config.id2label)

## 2. Orientation probe ← do not skip

These checkpoints ship with placeholder label names (`LABEL_0` / `LABEL_1`) and their cards do not
reliably state which index means *supported*. Get it wrong and every number below is silently
inverted — the worst kind of bug, because the notebook still runs and still prints plausible rates.

Worse, a checkpoint can load **with no missing-key warning at all** and still have an untrained
classification head, in which case it scores every claim alike and the "agreement" you measure is
agreement with noise. That is not hypothetical: it is exactly what the otherwise better FactCG
export does (see the config cell).

So this cell does not trust the model card. It scores twelve claims whose answers are not in dispute
against a document-length premise, **derives** the label index from the model's own ranking (by AUC,
which one outlier cannot flip), and then **refuses to continue** unless the separation is real.

In [ ]:
# Derive the label index from the model's own ranking on twelve claims whose answers are not in
# dispute, scored against a DOCUMENT-length premise (these checkpoints are trained on document-
# grounded fact checking and behave erratically on single sentences). Raises if the separation is
# not real — see nli_core.Verifier.probe.
_probe = VERIFIER.probe(threshold=THRESHOLD)
SUPPORTED_IDX, PROBE_AUC = _probe["supported_idx"], _probe["auc"]

def entail_probs(premises, hypotheses):
    """P(supported) per (premise, hypothesis) pair, in the orientation the probe established."""
    return VERIFIER.entail_probs(premises, hypotheses)

## 3. The claims to verify

The judgment logs written by nb9 / nb9b are already exactly the table this needs: one row per node
and per relation, carrying the **verbalised assertion** (the hypothesis) and the LLM judge's verdict.
Nothing is re-verbalised here, deliberately — both verifiers must see the *same* hypothesis string or
the comparison measures phrasing, not judgment.

Two classes of row are excluded from agreement, because the LLM never actually judged them and
scoring them against it would be comparing to a default rather than to a decision:

- **orphans** — relations auto-failed because an endpoint node was dropped;
- **unjudged** — in nb9 only, assertions whose evidence failed the deterministic quote check and so
  were failed before reaching the judge.

They are still scored and written out (the entailment model has no such gate, so its verdict on them
is informative — an orphan the article plainly supports is evidence the node-first cascade is too
aggressive), just held out of the agreement statistics.

In [ ]:
def load_article(article_id):
    return (ARTICLE_DIR / f"{article_id}.txt").read_text()

def claim_table(article_ids=None):
    """One row per (article, assertion), with each judge's verdict alongside. Read through
    nli_core so the cluster job scores exactly the same assertion strings for the same ids."""
    return [c for aid in (article_ids or ARTICLE_IDS)
            for c in nli_core.load_claims(RESULTS_DIR, aid, TRACK_DIR, SOURCES)]

CLAIMS = claim_table()
print(f"{len(CLAIMS)} assertion(s) across {len(ARTICLE_IDS)} article(s)")
for src in SOURCES:
    have   = [c for c in CLAIMS if f"{src}_grounded" in c]
    judged = [c for c in have if c[f"{src}_judged"]]
    print(f"  {src:9} {len(have):4} rows, {len(judged):4} actually judged by the LLM "
          f"({len(have)-len(judged)} orphaned/unjudged, held out of agreement)")
print("\nexample hypothesis:", CLAIMS[0]["assertion"][:100] if CLAIMS else "-")

## 4. Scoring — two granularities, on purpose

**Document pass (the headline).** The article is cut into overlapping token-budget chunks and the
claim is scored against each; the claim's score is the **max**. This mirrors nb9b's question exactly
— *is this supported anywhere in the article?* — so the comparison against nb9b's judge is
like-for-like.

**Localisation pass (`SENTENCE_PASS`).** The same claim is scored against every sliding window of
`WINDOW` consecutive sentences, and we keep the max **and the window that produced it**. That gives
the entailment model's own answer to "which sentence supports this?", directly comparable to nb9b's
`judge_quote` — a second, cheap check on the LLM's provenance that costs no API calls. Windows rather
than single sentences because support routinely spans a sentence boundary (a figure in one, its
subject in the next), and a single-sentence premise would fail those for the wrong reason.

The document score is what `nli_supported` uses. The localisation score is reported alongside; when
it is much lower than the document score, the support is genuinely distributed across the article
rather than sitting in one place — worth knowing about an assertion before you cite it.

Cost note: the localisation pass is roughly an order of magnitude more forward passes than the
document pass. Results are cached per article, so a re-run only scores what is new.

In [ ]:
# Scoring is nli_core.Verifier.score_claims — see section 4's text above for what the two passes
# are and why. This cell is just the per-article loop, the cache, and the write-out.
def verify(article_ids=None, force=None):
    force = FORCE_RESCORE if force is None else force
    config = {"model": VERIFIER_MODEL, "threshold": THRESHOLD, "window": WINDOW,
              "max_len": MAX_LEN, "sentence_pass": SENTENCE_PASS,
              "probe_auc": PROBE_AUC, "device": VERIFIER.device, "scored_on": "local"}
    out = {}
    for aid in (article_ids or ARTICLE_IDS):
        path   = RESULTS_DIR / aid / TRACK_DIR / nli_core.VERDICTS_NAME
        claims = [c for c in CLAIMS if c["article_id"] == aid]
        if path.exists() and not force:
            cached = json.loads(path.read_text())
            if cached.get("model") == VERIFIER_MODEL and len(cached.get("scores", {})) == len(claims):
                out[aid] = cached["scores"]
                where = cached.get("scored_on", "?")
                print(f"{aid:8} {len(claims):5} claims — cached (scored on {where})")
                continue
        t0 = time.perf_counter()
        scored = VERIFIER.score_claims(load_article(aid), [c["assertion"] for c in claims],
                                       window=WINDOW, sentence_pass=SENTENCE_PASS,
                                       threshold=THRESHOLD)
        by_id = {c["id"]: s for c, s in zip(claims, scored)}
        nli_core.write_verdicts(RESULTS_DIR, aid, TRACK_DIR, by_id, config)
        VERIFIER.free_gpu()                     # don't carry one article's peak into the next
        out[aid] = by_id
        n_sup = sum(s["nli_supported"] for s in scored)
        print(f"{aid:8} {len(claims):5} claims — {n_sup:5} supported "
              f"({n_sup/max(1,len(claims)):6.1%})  {time.perf_counter()-t0:6.1f}s")
    return out

SCORES = verify()

# attach to the claim rows, re-thresholding from the cached probabilities so changing THRESHOLD
# never requires re-scoring
for c in CLAIMS:
    s = SCORES.get(c["article_id"], {}).get(c["id"])
    if s:
        c.update(s)
        c["nli_supported"] = bool(s["nli_doc_score"] >= THRESHOLD)
print(f"\nscored {sum('nli_doc_score' in c for c in CLAIMS)}/{len(CLAIMS)} assertions")

## 5. Agreement — the number that strengthens the methodology

Raw agreement alone is misleading when one class dominates (both verifiers say "supported" ~90% of
the time, so a coin that always says yes scores ~90%). So this reports **Cohen's κ** alongside, which
corrects for agreement expected by chance, and the full 2×2 confusion so the *direction* of
disagreement is visible.

Read κ conservatively: 0.4–0.6 is moderate, 0.6–0.8 substantial. What matters for the thesis claim is
less the exact κ than the **consensus rate** — the share of assertions both verifiers independently
call supported. That is the number to report as your factuality figure, because it no longer depends
on one closed model's judgment.

In [ ]:
def _kappa(a, b):
    """Cohen's kappa for two binary label sequences."""
    n = len(a)
    if not n:
        return None
    obs = sum(x == y for x, y in zip(a, b)) / n
    pa, pb = sum(a) / n, sum(b) / n
    exp = pa * pb + (1 - pa) * (1 - pb)
    return None if exp == 1 else round((obs - exp) / (1 - exp), 3)

def agreement(source, rows=None):
    rows = [c for c in (rows or CLAIMS)
            if c.get(f"{source}_judged") and "nli_doc_score" in c]
    if not rows:
        return None
    llm = [bool(c[f"{source}_supported"]) for c in rows]
    nli = [bool(c["nli_supported"]) for c in rows]
    both_yes = sum(l and n for l, n in zip(llm, nli))
    both_no  = sum((not l) and (not n) for l, n in zip(llm, nli))
    llm_only = sum(l and not n for l, n in zip(llm, nli))
    nli_only = sum(n and not l for l, n in zip(llm, nli))
    n = len(rows)
    return {"n_judged": n,
            "llm_supported": sum(llm), "llm_rate": round(sum(llm) / n, 3),
            "nli_supported": sum(nli), "nli_rate": round(sum(nli) / n, 3),
            "raw_agreement": round((both_yes + both_no) / n, 3),
            "cohen_kappa": _kappa(llm, nli),
            "confusion": {"both_supported": both_yes, "both_rejected": both_no,
                          "llm_only": llm_only, "nli_only": nli_only},
            "consensus_supported_rate": round(both_yes / n, 3),
            "disagreements": llm_only + nli_only}

AGREEMENT = {"model": VERIFIER_MODEL, "threshold": THRESHOLD, "articles": ARTICLE_IDS,
             "overall": {}, "by_kind": {}, "by_article": {}}
for src in SOURCES:
    a = agreement(src)
    if not a:
        print(f"{src}: nothing judged to compare"); continue
    AGREEMENT["overall"][src] = a
    AGREEMENT["by_kind"][src] = {k: agreement(src, [c for c in CLAIMS if c["kind"] == k])
                                 for k in ("node", "relation")}
    AGREEMENT["by_article"][src] = {aid: agreement(src, [c for c in CLAIMS if c["article_id"] == aid])
                                    for aid in ARTICLE_IDS}
    print(f"=== vs {src} judge ".ljust(60, "="))
    print(f"  judged assertions      {a['n_judged']}")
    print(f"  LLM says supported     {a['llm_supported']:4}  ({a['llm_rate']:.1%})")
    print(f"  NLI says supported     {a['nli_supported']:4}  ({a['nli_rate']:.1%})")
    print(f"  raw agreement          {a['raw_agreement']:.1%}      Cohen's kappa {a['cohen_kappa']}")
    c = a["confusion"]
    print(f"  both supported {c['both_supported']:4}   both rejected {c['both_rejected']:4}   "
          f"LLM only {c['llm_only']:4}   NLI only {c['nli_only']:4}")
    print(f"  --> CONSENSUS supported: {a['consensus_supported_rate']:.1%} "
          f"({c['both_supported']}/{a['n_judged']}) — the defensible factuality number\n")

(GLOBAL_DIR / "nli_agreement.json").write_text(json.dumps(AGREEMENT, indent=2))
print("wrote", (GLOBAL_DIR / "nli_agreement.json").relative_to(ROOT))

## 6. The disagreements — your human-reading queue

This is the practical payoff. Instead of sampling 50 assertions at random to check the judge, you now
have the *specific* set where two independent verifiers of different kinds disagree. That set is
small, and it is enriched for genuinely hard or genuinely wrong cases — reading it is a far better use
of an hour than reading 50 random supported assertions.

Each row carries both verdicts, the LLM's justification, the entailment model's probability, and the
sentence **it** thought was the best support — so you can usually adjudicate without re-reading the
whole article. Fill `human` with `y`/`n` and section 7 will tell you which verifier was right.

In [ ]:
import csv

def disagreement_sheet(source):
    rows = [c for c in CLAIMS
            if c.get(f"{source}_judged") and "nli_doc_score" in c
            and bool(c[f"{source}_supported"]) != bool(c["nli_supported"])]
    rows.sort(key=lambda c: abs(c["nli_doc_score"] - THRESHOLD), reverse=True)   # most clear-cut first
    return [{"article_id": c["article_id"], "kind": c["kind"], "id": c["id"],
             "assertion": c["assertion"],
             "llm_verdict": "supported" if c[f"{source}_supported"] else "rejected",
             "llm_reason": c.get(f"{source}_note"),
             "llm_quote": c.get("judge_quote"),
             "nli_verdict": "supported" if c["nli_supported"] else "rejected",
             "nli_score": c["nli_doc_score"],
             "nli_best_window": c.get("nli_best_window"),
             "who_is_more_confident": ("nli" if abs(c["nli_doc_score"] - THRESHOLD) > 0.4 else "unclear"),
             "human": ""} for c in rows]

for src in SOURCES:
    sheet = disagreement_sheet(src)
    path  = GLOBAL_DIR / f"nli_disagreements_{src}.csv"
    cols  = list(sheet[0]) if sheet else ["article_id", "kind", "id", "assertion", "human"]
    with path.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=cols); w.writeheader(); w.writerows(sheet)
    (GLOBAL_DIR / f"nli_disagreements_{src}.json").write_text(
        json.dumps(sheet, ensure_ascii=False, indent=2))
    print(f"{src}: {len(sheet)} disagreement(s) -> {path.relative_to(ROOT)}")
    for r in sheet[:6]:
        print(f"\n  [{r['article_id']} {r['kind']}] {r['assertion'][:88]}")
        print(f"     LLM {r['llm_verdict']:9} : {str(r['llm_reason'])[:88]}")
        print(f"     NLI {r['nli_verdict']:9} : P={r['nli_score']:.3f}")
        if r["nli_best_window"]:
            print(f"     NLI best support : {r['nli_best_window'][:88]}")

## 7. Calibrate the threshold (optional, but do it once)

`THRESHOLD = 0.5` is an arbitrary default, and the operating point genuinely matters: it trades the
two verifiers' disagreement rate against each other. If you have labelled the blinded sample from
§5 of nb9/nb9b, this cell reuses those labels as ground truth and sweeps the threshold, reporting
**balanced accuracy** (the LLM-AggreFact metric, so your number is comparable to the published
leaderboard) for both verifiers on the same items.

That comparison is the strongest single result this notebook can give you: *on human-labelled data,
here is how the closed LLM judge and the open encoder each score* — which turns "we trust the judge"
into a measured claim. If no labels exist yet, the cell says so and does nothing.

In [ ]:
def _bacc(gold, pred):
    tp = sum(g and p for g, p in zip(gold, pred)); fn = sum(g and not p for g, p in zip(gold, pred))
    tn = sum((not g) and (not p) for g, p in zip(gold, pred)); fp = sum((not g) and p for g, p in zip(gold, pred))
    if not (tp + fn) or not (tn + fp):
        return None
    return round(0.5 * (tp / (tp + fn) + tn / (tn + fp)), 3)

def calibrate(source):
    """Use the human-labelled sample from nb9/nb9b §5 as ground truth on the items it covers."""
    stem = "grounding_judge" if source == "evidence" else "grounding_fulltext_judge"
    sp, kp = GLOBAL_DIR / f"{stem}_sample.json", GLOBAL_DIR / f"{stem}_key.json"
    if not sp.exists():
        print(f"{source}: no labelled sample at {sp.relative_to(ROOT)} — run §5 of the matching "
              "notebook, label it, then re-run this cell")
        return
    labelled = json.loads(sp.read_text())
    key      = json.loads(kp.read_text())
    by_id    = {(c["article_id"], c["id"]): c for c in CLAIMS}
    gold, llm, prob = [], [], []
    for x in labelled:
        h = (x.get("human") or "").strip().lower()
        c = by_id.get((x.get("article_id"), x.get("id")))
        if h not in ("y", "n") or not c or "nli_doc_score" not in c:
            continue
        gold.append(h == "y"); llm.append(bool(key[str(x["index"])])); prob.append(c["nli_doc_score"])
    if len(gold) < 10:
        print(f"{source}: only {len(gold)} labelled item(s) matched — label more before trusting this")
        return
    print(f"=== {source}: {len(gold)} human-labelled item(s), {sum(gold)} supported ".ljust(60, "="))
    print(f"  LLM judge          balanced accuracy {_bacc(gold, llm)}")
    print(f"  {'thr':>5}  {'bacc':>6}  {'agree w/ human':>15}")
    best = None
    for t in [i / 20 for i in range(1, 20)]:
        pred = [p >= t for p in prob]
        b = _bacc(gold, pred)
        if b is None:
            continue
        agree = sum(g == p for g, p in zip(gold, pred)) / len(gold)
        print(f"  {t:>5.2f}  {b:>6}  {agree:>14.1%}")
        if best is None or b > best[1]:
            best = (t, b)
    if best is None:
        print("  the labelled sample is single-class — cannot calibrate on it"); return
    print(f"\n  best threshold {best[0]:.2f} (balanced accuracy {best[1]}) — set THRESHOLD to this "
          "in the config cell and re-run from section 4 (cached scores, no re-scoring)")

for src in SOURCES:
    calibrate(src)